# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/01/05 19:57:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/01/05 19:57:25 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
23/01/05 19:57:32 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


In [5]:
year0 = "2019"

In [6]:
year1 = str(int(year0) + 1)
#year1 = year0

# Set Parameters

Set date range and airports here.

In [7]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 + "-01-01"}

In [8]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

In [9]:
#airports = [ "KADW", ]

# UDFs

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [10]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

Define the schema of the litetrack points structure array

In [11]:
points_schema = T.StructType([
    T.StructField("points", T.ArrayType(
        T.StructType([
            T.StructField("primary_key", T.StringType(), True),
            T.StructField("time", T.LongType(), True),
            T.StructField("latitude", T.DoubleType(), True),
            T.StructField("longitude", T.DoubleType(), True),
            T.StructField("altitude", T.FloatType(), True),
            T.StructField("course", T.FloatType(), True),
            T.StructField("speed", T.FloatType(), True),
            T.StructField("source_point_keys", T.ArrayType(
                T.StructType([
                    T.StructField("element", T.StringType(), True),
                ]), True), True),
            T.StructField("acceleration", T.DoubleType(),True),
            T.StructField("type", T.StringType(), True),
            T.StructField("airspace_key", T.StringType(), True),
            T.StructField("tas", T.FloatType(), True),
            T.StructField("ias", T.FloatType(), True),
            T.StructField("along_track_distance", T.FloatType(), True),
            T.StructField("derived_point_key", T.StringType(), True),
        ]), True), True)
    ])

Define a function that will convert the points array into a string containing n-tuples of data for each litetrack point

In [12]:
none_string = "*"
separator_string = ":"

def stringify_points(array_of_points:T.ArrayType(points_schema)) -> str:
    output = ""
    for x in array_of_points:
        output += (none_string if x.time is None else str(x.time)) + " "
        output += (none_string if x.latitude is None else f'{x.latitude:.6f}') + " "
        output += (none_string if x.longitude is None else f'{x.longitude:.6f}') + " "
        output += (none_string if x.altitude is None else f'{x.altitude:.0f}') + " "
        output += (none_string if x.course is None else f'{x.course:.0f}') + " "
        output += (none_string if x.speed is None else f'{x.speed:.0f}') + separator_string

    # drop the trailing tuple separator
    if len(array_of_points) > 0:
        output = output[0:len(separator_string) * -1]
    return output

stringify_points_udf = F.udf(stringify_points, T.StringType())

# Load LiteTrack

the points array is "stringified" to a new `trackpoints` column

In [13]:
# get litetracks for the date range of interest

df_litetracks = (
    api.dataframe("LiteTrack", **dates, partition_filters=api.custom_partitions("ASSOCIATED"))
    .select(
        "track_key",
        F.col("start_time").alias("start_epoch_millisec"),
        F.col("end_time").alias("end_epoch_millisec"),
        "points",
    )
)

Could not find data for: 2019-01-30, 2019-02-14
Multiple versions found: 3.1.27, 3.1.30


# Load PlannedRoute

In [15]:
## get the plannedroutes for tracks to and from airports of interest

df_plannedroutes = (
    api.dataframe("PlannedRoute", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .select(
        "track_key",
        F.col("departure_aerodrome").alias("orig"),
        F.col("destination_aerodrome").alias("dest"),
    )
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
    .drop("orig", "dest")
)

Multiple versions found: 3.1.24, 3.1.34, 3.1.40


# Load FlightplanSeries

In [18]:
## get the flightplans for tracks to and from airports of interest

df_flightplanseries = (
    api.dataframe("FlightplanSeries", **dates, metadata=True)
    .select(
        "track_key",
        to_date("metadata.effective_start_date").alias("start_date"),
        to_date("metadata.effective_end_date").alias("end_date"),
        "callsign",
        "aircraft_type",
        "mode_s_code",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
    )
    .withColumn("month", F.substring("start_date", 5, 2))
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
)

Multiple versions found: 3.1.20, 3.1.34


# Join the dataframes as needed

In [19]:
df_temp = df_litetracks.join(df_flightplanseries, on=["track_key"], how="inner")

In [20]:
df_joined = df_temp.join(df_plannedroutes, on=["track_key"], how="inner")

# Create Arrivals and Departures dataframes and Combine to one

In [21]:
df_arrivals = (
    df_joined.withColumn("airport", F.col("dest"))
    .withColumn("operation", F.lit("ARRIVALS"))
    .filter(F.col("airport").isin(airports))
)

In [22]:
df_departures = (
    df_joined.withColumn("airport", F.col("orig"))
    .withColumn("operation", F.lit("DEPARTURES"))
    .filter(F.col("airport").isin(airports))
)

In [23]:
df_litetracks_combined = df_arrivals.unionByName(df_departures)

# Save LiteTracks Data

In [24]:
df_litetracks_output = (
    df_litetracks_combined
    .withColumn("trackpoints", stringify_points_udf("points"))
    .select(
        "track_key",
        "start_date",
        "end_date",
        "orig",
        "dest",
        "mode_s_code",
        "callsign",
        "aircraft_type",
        "trackpoints",
        "airport",
        "operation",
        "month",
    )
)

In [25]:
#df_litetracks_output.show()

In [26]:
(
    df_litetracks_output
    .repartition("airport", "operation", "month")
    .write.option("header", True).partitionBy(["airport", "operation", "month"])
    .csv("CRAFT/" + year0 + "/litetracks", compression="gzip", mode="overwrite")
)

23/01/05 20:10:23 WARN TaskSetManager: Lost task 406.0 in stage 31.0 (TID 11594) (hdp1-dn135.mitre.org executor 82): FetchFailed(BlockManagerId(62, hdp1-dn74.mitre.org, 7337, None), shuffleId=0, mapIndex=99, mapId=99, reduceId=406, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark

23/01/05 20:10:24 WARN TaskSetManager: Lost task 383.0 in stage 31.0 (TID 11571) (hdp1-dn93.mitre.org executor 217): FetchFailed(BlockManagerId(62, hdp1-dn74.mitre.org, 7337, None), shuffleId=0, mapIndex=99, mapId=99, reduceId=383, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark

23/01/05 20:10:24 WARN TaskSetManager: Lost task 430.0 in stage 31.0 (TID 11618) (hdp1-dn120.mitre.org executor 228): FetchFailed(null, shuffleId=0, mapIndex=-1, mapId=-1, reduceId=430, message=
org.apache.spark.shuffle.MetadataFetchFailedException: Missing an output location for shuffle 0 partition 430
	at org.apache.spark.MapOutputTracker$.validateStatus(MapOutputTracker.scala:1701)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$10(MapOutputTracker.scala:1648)
	at org.apache.spark.MapOutputTracker$.$anonfun$convertMapStatuses$10$adapted(MapOutputTracker.scala:1647)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:943)
	at scala.collection.AbstractIterator.foreach(Iterator.scala:1431)
	at org.apache.spark.MapOutputTracker$.convertMapStatuses(MapOutputTracker.scala:1647)
	at org.apache.spark.MapOutputTrackerWorker.getMapSizesByExecutorIdImpl(MapOutputTracker.scala:1290)
	at org.apache.spark.MapOutputTrac

23/01/05 20:10:25 WARN TaskSetManager: Lost task 410.0 in stage 31.0 (TID 11598) (hdp1-dn63.mitre.org executor 114): FetchFailed(BlockManagerId(62, hdp1-dn74.mitre.org, 7337, None), shuffleId=0, mapIndex=99, mapId=99, reduceId=410, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark

23/01/05 20:10:25 WARN TaskSetManager: Lost task 385.0 in stage 31.0 (TID 11573) (hdp1-dn135.mitre.org executor 219): FetchFailed(BlockManagerId(62, hdp1-dn74.mitre.org, 7337, None), shuffleId=0, mapIndex=99, mapId=99, reduceId=385, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spar

...

23/01/05 20:44:49 WARN TaskSetManager: Lost task 119.0 in stage 40.0 (TID 14747) (hdp1-dn01.mitre.org executor 389): FetchFailed(BlockManagerId(185, hdp1-dn74.mitre.org, 7337, None), shuffleId=9, mapIndex=368, mapId=368, reduceId=308, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.sp

23/01/05 20:44:50 WARN TaskSetManager: Lost task 26.0 in stage 40.0 (TID 14654) (hdp1-dn55.mitre.org executor 251): FetchFailed(BlockManagerId(185, hdp1-dn74.mitre.org, 7337, None), shuffleId=9, mapIndex=368, mapId=368, reduceId=81, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spar

23/01/05 20:44:50 WARN TaskSetManager: Lost task 81.0 in stage 40.0 (TID 14709) (hdp1-dn29.mitre.org executor 351): FetchFailed(BlockManagerId(185, hdp1-dn74.mitre.org, 7337, None), shuffleId=9, mapIndex=368, mapId=368, reduceId=227, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spa

23/01/05 20:44:51 WARN TaskSetManager: Lost task 85.0 in stage 40.0 (TID 14713) (hdp1-dn22.mitre.org executor 411): FetchFailed(BlockManagerId(62, hdp1-dn74.mitre.org, 7337, None), shuffleId=9, mapIndex=418, mapId=418, reduceId=235, message=
org.apache.spark.shuffle.FetchFailedException
	at org.apache.spark.errors.SparkCoreErrors$.fetchFailedError(SparkCoreErrors.scala:312)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.throwFetchFailedException(ShuffleBlockFetcherIterator.scala:1166)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:904)
	at org.apache.spark.storage.ShuffleBlockFetcherIterator.next(ShuffleBlockFetcherIterator.scala:85)
	at org.apache.spark.util.CompletionIterator.next(CompletionIterator.scala:29)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:486)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:492)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spar

23/01/05 20:45:48 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_32_piece0 !
